# Zoom — Spectral Clustering — Breast Cancer Wisconsin (Diagnostic)

**Course:** UE 5BM753 — Supervised and Unsupervised Learning for Biology
**Part of:** the Breast Cancer Wisconsin (Diagnostic) final report, split into
per-step notebooks. See `00_Final_Report_.ipynb` for the combined narrative and
`README.md` for the overall conclusion.

**Question addressed here:** do the data naturally separate into two clusters that match the known diagnosis, without ever using the labels?

Tunes and evaluates Spectral Clustering more closely, restricted to the features selected in `05_Feature_Selection.ipynb`.

Spectral Clustering was the strongest unsupervised baseline in the initial
screening, based on its match to the true diagnosis (ARI). This notebook tunes and
evaluates it more closely, in three steps:
- **Choosing the number of clusters**: silhouette score is swept over a range of
  cluster counts, since Spectral Clustering has no centroids and no elbow/inertia to
  rely on.
- **Fitting the final clustering**: the best cluster count is used to refit the
  model, and the result is compared to the true diagnosis using the same external
  metrics as the screening (ARI, homogeneity, completeness, V-measure).
- **Comparison: all features vs. selected features**: the resulting match to the
  diagnosis is compared to the screening's result on all 30 features, to check
  whether the feature reduction helps or hurts the natural clustering structure.

### Choosing the number of clusters

Spectral Clustering has no centroids, so there is no elbow/inertia to rely on for
picking the number of clusters. Silhouette score is swept over a range instead,
using the features selected earlier.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

sys.path.append(os.path.join("..", "01_scripts"))
from dataset import load_data, get_feature_columns
from cluster_selection import run_spectral_range
from plots import plot_silhouette_by_k

# This notebook only needs the files exported by earlier notebooks, not their
# in-memory variables, so it can run entirely on its own.
RANDOM_STATE = 42
DATA_PATH = os.path.join("..", "02_data", "raw", "10_Breast_Cancer_Wisconsin_Diagnostic.csv")
RESULTS_DIR = os.path.join("..", "04_results")
FIGURES_DIR = os.path.join("..", "05_figures")
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

df = load_data(DATA_PATH)
features = get_feature_columns(df)

# Feature names only, not the train/test CSVs: those cover only part of the
# samples and use a train-only scaler. Clustering uses all 569 samples instead.
selected_features = pd.read_csv(os.path.join(RESULTS_DIR, "03_selected_features.csv"))["feature"].tolist()

X_scaled_selected = StandardScaler().fit_transform(df[selected_features].values)

spectral_k_results = run_spectral_range(X_scaled_selected, k_range=range(2, 8), random_state=RANDOM_STATE)
spectral_k_results.to_csv(os.path.join(RESULTS_DIR, "07_spectral_k_selection.csv"), index=False)

plot_silhouette_by_k(
    spectral_k_results,
    os.path.join(FIGURES_DIR, "final_10_spectral_k_selection.png"),
    title="Spectral Clustering: silhouette by number of clusters (selected features)",
)

best_k = int(spectral_k_results.loc[spectral_k_results["silhouette"].idxmax(), "k"])
print(f"Optimal n_clusters by silhouette: {best_k}")
display(spectral_k_results.round(4))

**Findings.** Silhouette peaks at k = 2 (0.399), well above every other value
tested (0.16 to 0.19 for k = 3 to 7). The clustering naturally recovers 2 groups on
the selected features alone, without ever using the diagnosis, matching the known
two-class structure of the data.

### Fitting the final clustering

The cluster count that reached the best silhouette score above is used to refit
Spectral Clustering. Its match to the true diagnosis is measured with the same
metrics as the initial screening (ARI, homogeneity, completeness, V-measure), even
though the diagnosis is never used to fit the clustering itself. A PCA is
recomputed on the same selected features, only to place each point on a 2D graph.
It has no role in the clustering itself.

In [ ]:
from sklearn.cluster import SpectralClustering
from evaluation import evaluate_clusterers
from plots import plot_pca_projection, plot_cluster_vs_diagnosis

spectral_final = SpectralClustering(n_clusters=best_k, affinity="nearest_neighbors", random_state=RANDOM_STATE)

spectral_eval = evaluate_clusterers(
    {"Spectral (selected features)": spectral_final}, X_scaled_selected, df["label"].values
)
spectral_eval.to_csv(os.path.join(RESULTS_DIR, "07_spectral_final_metrics.csv"), index=False)

print(f"--- Spectral Clustering (n_clusters={best_k}, selected features) vs. true diagnosis ---")
display(spectral_eval.round(4))

# PCA recomputed on the selected features only: gives the 2D coordinates for the
# plot below, but plays no role in the clustering itself.
pca_selected, X_pca_selected = plot_pca_projection(
    X_scaled_selected, df["label"].values,
    os.path.join(FIGURES_DIR, "final_14_pca_projection_selected_features.png"),
    random_state=RANDOM_STATE,
)

# Re-fit to get the cluster labels for the comparison figure below --
# evaluate_clusterers only returns summary metrics, not the labels themselves.
spectral_labels = spectral_final.fit_predict(X_scaled_selected)
plot_cluster_vs_diagnosis(
    X_pca_selected, df["label"].values, spectral_labels,
    os.path.join(FIGURES_DIR, "final_15_spectral_vs_diagnosis_selected.png"),
    title="True diagnosis vs. Spectral Clustering (selected features)"
)

**Findings.** Refit with 2 clusters, Spectral Clustering reaches an ARI of 0.792
against the true diagnosis, with similar homogeneity (0.697) and completeness
(0.730). This confirms the natural groups found on the 7 selected features largely
coincide with benign vs. malignant, even though the clustering never saw the
labels.

### Comparison: all features vs. selected features

Spectral Clustering was already evaluated on all 30 features during the initial
screening. Comparing that result to the one obtained here, on the reduced feature
set, shows whether removing redundant and uninformative features helps or hurts how
well the natural clusters match the true diagnosis.

In [ ]:
# All-30-features result: already computed in the initial screening.
screening_eval = pd.read_csv(os.path.join(RESULTS_DIR, "04_clustering_eval.csv"))
spectral_all_features = screening_eval[screening_eval["model"] == "Spectral"].iloc[0]

comparison = pd.DataFrame([
    {"features": "All 30 features", "n_features": len(features),
     "ARI_vs_diagnosis": spectral_all_features["ARI_vs_diagnosis"],
     "silhouette": spectral_all_features["silhouette"]},
    {"features": "Selected features", "n_features": len(selected_features),
     "ARI_vs_diagnosis": spectral_eval["ARI_vs_diagnosis"].iloc[0],
     "silhouette": spectral_eval["silhouette"].iloc[0]},
])
comparison.to_csv(os.path.join(RESULTS_DIR, "07_spectral_feature_comparison.csv"), index=False)

print("--- Spectral Clustering: all features vs. selected features ---")
display(comparison.round(4))

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(comparison["features"], comparison["ARI_vs_diagnosis"], color=["#8c8c8c", "#C44E52"])
ax.set_ylabel("ARI vs. true diagnosis")
ax.set_title("Spectral Clustering: feature reduction effect on ARI")
fig.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, "final_16_spectral_feature_comparison.png"))
plt.show()

**Findings.** Restricting to the 7 selected features improves both scores compared
to all 30 features: ARI rises from 0.761 to 0.792, and silhouette from 0.337 to
0.399. Removing redundant and less informative features does not only help the
supervised model; it also makes the natural clustering structure clearer.

## Discussion

Spectral Clustering was chosen for this zoom because it reached the highest ARI
against the true diagnosis in the initial screening, unlike Agglomerative
(average/complete), which collapsed into one dominant cluster despite a high
silhouette score. Restricting to the 7 selected features improves both its ARI
(0.761 to 0.792) and its silhouette (0.337 to 0.399) compared to using all 30
features.

**Limitation.** Only `n_clusters` was swept here; the affinity (`nearest_neighbors`)
and its other parameters were fixed, so a different choice could change the
result.